# Cross-Day Generalization 2014 
### CIC-IDS2017 | Author: Shanzay Jamil

## Why this experiment exists

The main notebook reports ~99.9% accuracy using a **random 80/20 split**. That number is real,
but it answers an *easy* question: *"Can the model recognize attacks from campaigns it has
already seen?"* Train and test flows come from the same days, the same attack runs, the same
network conditions 2014 the test set is statistically a mirror of the training set.

A **deployed IDS never gets that luxury.** It must catch attacks on days it never saw, launched
by tools it never saw. The realistic question is:

> **Can a model trained on some days detect attacks on a completely different day 2014
> including attack types it has never seen?**

This notebook answers that with two experiments of increasing difficulty:

| Experiment | Train on | Test on | What it measures |
|---|---|---|---|
| **1. Temporal split** | Tuesday + Wednesday | Friday (DDoS + PortScan) | New day, new campaigns; PortScan is an *unseen* attack family |
| **2. Unseen attack family** | Tue + Wed + Fri | Thursday (Web Attacks) | The hardest case: attacks that hide inside normal-looking HTTP |

**Expectation going in:** scores should drop versus the 99.9% baseline — and *that drop is
the finding*. It quantifies the generalization gap, which is the actual research problem in
ML-based intrusion detection.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, accuracy_score,
                             confusion_matrix, f1_score, recall_score,
                             precision_score)

DATA_DIR = 'data/'

FILES = {
    'tuesday':   DATA_DIR + 'Tuesday-WorkingHours.pcap_ISCX.csv',            # Brute Force (FTP/SSH)
    'wednesday': DATA_DIR + 'Wednesday-workingHours.pcap_ISCX.csv',          # DoS (Hulk, GoldenEye, Slowloris...)
    'thursday':  DATA_DIR + 'Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv',  # XSS, SQLi, Web Brute Force
    'friday_ddos': DATA_DIR + 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv',      # DDoS (LOIC)
    'friday_scan': DATA_DIR + 'Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv',  # PortScan
}

SAMPLE_PER_FILE = 50000   # None = load everything (needs RAM)

## Shared preprocessing

Identical cleaning to the main notebook, wrapped in a function so both experiments use
**exactly the same treatment** — the only variable we change is *which days* go where.
Changing one thing at a time is what makes the comparison valid (a controlled experiment).

Note the label conversion is case/whitespace-proof and keeps a copy of the original attack
name in `AttackType` so we can break results down per attack later.

In [ ]:
def load_and_clean(file_list, sample=SAMPLE_PER_FILE):
    """Load CSVs -> one clean DataFrame with binary Label + original AttackType."""
    dfs = []
    for f in file_list:
        t = pd.read_csv(f, low_memory=False)
        t.columns = t.columns.str.strip()
        if sample is not None and len(t) > sample:
            t = t.sample(n=sample, random_state=42)
        dfs.append(t)
    d = pd.concat(dfs, ignore_index=True)

    d.replace([np.inf, -np.inf], np.nan, inplace=True)
    d.dropna(inplace=True)
    d.drop_duplicates(inplace=True)

    d['AttackType'] = d['Label'].astype(str).str.strip()          # keep original name
    d['Label'] = (d['AttackType'].str.upper() != 'BENIGN').astype(int)
    return d

def make_xy(d):
    X = d.drop(columns=['Label']).select_dtypes(include=[np.number])
    y = d['Label']
    return X, y

def evaluate(name, y_true, y_pred):
    acc  = accuracy_score(y_true, y_pred)
    rec  = recall_score(y_true, y_pred)        # attack recall: attacks caught
    prec = precision_score(y_true, y_pred)     # flagged-as-attack correctness
    f1   = f1_score(y_true, y_pred)
    print(f'--- {name} ---')
    print(f'Accuracy: {acc*100:.2f}% | Attack Recall: {rec*100:.2f}% | '
          f'Precision: {prec*100:.2f}% | F1: {f1:.4f}')
    print(classification_report(y_true, y_pred, labels=[0,1],
                                target_names=['BENIGN','ATTACK'], zero_division=0))
    return {'Accuracy': acc, 'Attack Recall': rec, 'Precision': prec, 'F1': f1}

## Experiment 1 — Train Tue+Wed, test Friday

**What the model knows:** Brute-Force attacks (Tuesday) and DoS attacks (Wednesday).
**What it faces:** Friday's DDoS (a *distributed* variant of DoS — related but not identical)
and **PortScan — an attack family it has never seen at all.**

Also note there is **no random split** here. Train and test are entirely different files,
different days. We align on the intersection of feature columns to be robust to any small
schema differences between file releases.

In [ ]:
train_df = load_and_clean([FILES['tuesday'], FILES['wednesday']])
test_df  = load_and_clean([FILES['friday_ddos'], FILES['friday_scan']])

print('Train label mix:'); print(train_df['AttackType'].value_counts(), '\n')
print('Test label mix:');  print(test_df['AttackType'].value_counts())

X_train, y_train = make_xy(train_df)
X_test,  y_test  = make_xy(test_df)

# Use only features present in both (protects against schema drift between files)
common = X_train.columns.intersection(X_test.columns)
X_train, X_test = X_train[common], X_test[common]
print(f'\nUsing {len(common)} common features')

rf1 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf1.fit(X_train, y_train)
res1 = evaluate('Experiment 1: Tue+Wed -> Friday', y_test, rf1.predict(X_test))

### Per-attack breakdown — where exactly does it fail?

Aggregate recall hides the interesting part. Here we compute recall **per attack type** in the
test day: e.g., does the model catch DDoS (similar to the DoS it trained on) but miss PortScan
(never seen)?

In [ ]:
pred1 = rf1.predict(X_test)
per_attack = (pd.DataFrame({'AttackType': test_df['AttackType'].values,
                            'true': y_test.values, 'pred': pred1})
                .query('true == 1')
                .groupby('AttackType')['pred'].agg(['mean','count'])
                .rename(columns={'mean':'recall','count':'n_samples'})
                .sort_values('recall'))
print(per_attack)

per_attack['recall'].plot(kind='barh', figsize=(8,4), color='#e67e22')
plt.title('Experiment 1 — Recall per unseen attack type', fontweight='bold')
plt.xlabel('Recall (fraction of attacks caught)')
plt.xlim(0,1); plt.tight_layout()
plt.savefig('exp1_per_attack_recall.png', dpi=150); plt.show()

## Experiment 2 — The hardest case: unseen Web Attacks

**Train on:** Tuesday + Wednesday + both Friday files (Brute Force, DoS, DDoS, PortScan).
**Test on:** Thursday's Web Attacks (XSS, SQL Injection, Web Brute Force) + Thursday's benign traffic.

Why this is the hardest test in the dataset:
1. The attack family is completely absent from training.
2. Web attacks are **content-based** — the maliciousness lives in the HTTP payload text
   (the injected SQL/JavaScript), but our features only describe traffic *shape*
   (sizes, timing, flags). An SQLi request has nearly the same shape as a normal form submit.
3. Very few attack samples exist even in the test day → recall estimates are noisy.

In [ ]:
train_df2 = load_and_clean([FILES['tuesday'], FILES['wednesday'],
                            FILES['friday_ddos'], FILES['friday_scan']])
test_df2  = load_and_clean([FILES['thursday']])

print('Test label mix:'); print(test_df2['AttackType'].value_counts())

X_train2, y_train2 = make_xy(train_df2)
X_test2,  y_test2  = make_xy(test_df2)
common2 = X_train2.columns.intersection(X_test2.columns)
X_train2, X_test2 = X_train2[common2], X_test2[common2]

rf2 = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf2.fit(X_train2, y_train2)
res2 = evaluate('Experiment 2: all other days -> Thursday Web Attacks', y_test2, rf2.predict(X_test2))

## Putting it together — the generalization gap

The chart below is the punchline of the whole repository: the same model family, the same
features, the same cleaning — only the *evaluation honesty* changes.

*(After running, replace the `baseline` numbers with your actual within-day results from the
main notebook.)*

In [ ]:
baseline = {'Accuracy': 0.9990, 'Attack Recall': 0.999, 'F1': 0.9983}   # from main notebook

summary = pd.DataFrame({
    'Within-day (random 80/20)': baseline,
    'Cross-day: -> Friday':      {k: res1[k] for k in baseline},
    'Cross-day: -> Thu WebAtk':  {k: res2[k] for k in baseline},
}).T

print(summary.round(4))

summary.plot(kind='bar', figsize=(10,5), color=['#3498db','#e67e22','#2ecc71'])
plt.title('The Generalization Gap — same model, honest vs. easy evaluation', fontweight='bold')
plt.ylabel('Score'); plt.ylim(0,1.05); plt.xticks(rotation=15, ha='right')
plt.legend(loc='lower right'); plt.tight_layout()
plt.savefig('generalization_gap.png', dpi=150); plt.show()

## How to read the results

- **If cross-day scores stay high** → flow-shape features genuinely transfer across days for
  volumetric attacks. Expected for DDoS/PortScan: even unseen, a port scan's traffic shape
  (thousands of tiny one-sided flows) is unlike anything benign.
- **If Web-Attack recall collapses** (it very likely will) → this is the honest headline finding:
  *flow features are sufficient for volumetric attacks but structurally blind to content-based
  attacks.* No amount of extra flow data fixes this — it motivates payload-aware features,
  NLP over HTTP bodies, and graph/behavioural representations (which is precisely the direction
  of current research, e.g., behavioural graph representation for stealthy threat profiling).
- **Watch precision too, not just recall.** A new day's *benign* traffic also differs; if
  precision drops, the model is raising false alarms on unfamiliar-but-innocent traffic —
  the other face of the generalization problem.

## Honest limitations of this experiment

1. Both "days" still come from the **same lab network** (same topology, same simulated users).
   True deployment shift — different network, different era — would be harder still
   (e.g., testing on CSE-CIC-IDS2018).
2. `SAMPLE_PER_FILE` subsampling means per-attack recalls on rare classes are noisy.
3. Binary labels: we measure attack-vs-benign transfer, not attack-type identification.
4. No hyperparameter tuning — deliberately, so the comparison isolates the *data split* effect.